In [1]:
from pyspark.sql import SparkSession
import getpass


username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.config("spark.shuffle.service.enabled", "false") \
.config("spark.dynamicAllocation.enabled", "false") \
.config("spark.executor.instances", "3") \
.enableHiveSupport() \
.master("yarn") \
.appName("025320_cachin2_2") \
.getOrCreate()

In [3]:
load_file = spark.read.parquet("external_tables/order/part-00000-527264b6-6f59-4738-8ca9-853669a79483-c000.snappy.parquet")

In [4]:
load_file.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
# caching_250320.order_external_caching
spark.sql("""
    CREATE TABLE caching_250320.2_caching_order_external
    (order_id int, order_date timestamp, customer_id int, order_status string)
    USING PARQUET
    LOCATION '/user/itv025320/external_tables/order'
""")

""


In [6]:
# spark.sql("DROP TABLE caching_250320.2_caching_order_external")

In [7]:
# spark.sql("SELECT * FROM caching_250320.2_caching_order_external limit 5").show()

In [8]:
spark.sql("describe extended caching_250320.2_caching_order_external").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |int                                                               |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |int                                                               |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |caching_250320  

In [9]:
import time
start_time = time.time()
spark.sql("SELECT COUNT(*) FROM caching_250320.2_caching_order_external").show(5)
print(f"Time to load result without caching: {time.time() - start_time} seconds")

+--------+
|count(1)|
+--------+
|   68883|
+--------+

Time to load result without caching: 1.0143110752105713 seconds


In [10]:
start_time = time.time()
spark.sql("CACHE TABLE caching_250320.2_caching_order_external")
print(f"Time to cache table: {time.time() - start_time} seconds")

Time to cache table: 2.025885581970215 seconds


In [11]:
start_time = time.time()
spark.sql("SELECT COUNT(*) FROM caching_250320.2_caching_order_external").show(5)
print(f"Time to load result After caching: {time.time() - start_time} seconds")

+--------+
|count(1)|
+--------+
|   68883|
+--------+

Time to load result After caching: 0.26189684867858887 seconds


In [12]:
# spark.sql("uncache table caching_250320.2_caching_order_external")